# Suo - step 1: AnnData preparation

This notebook computes everything about `suo` that is expensive and
stable, and writes it once to an intermediate `.h5ad`:

- the **cell subset** the published dataset is made of, applied before anything
  else so the graph and the three layouts see exactly the published cells;
- one shared k-nearest-neighbour graph on the scANVI latent representation;
- 1D, 2D and 3D UMAP embeddings, all three from **that one graph**, so the three
  views of the dataset encode identical neighbour relationships;
- the latent representation the viewer uses for centroid statistics;
- the curated cell metadata.

Step 2 (`suo_2_export_cellucid.ipynb`) reads only the intermediate, resolves
the gene names, and calls `cellucid.prepare`. The split exists so that a change to
the export format, to the gene naming, or to the `prepare` API costs one cheap
re-run of step 2 instead of a full re-embedding.

**This notebook establishes no gene names.** The source `var` index is the
positional matrix column number and nothing else; the identifiers live in
`.anndatas/varnames/dataset_complete_Suo_varnames.pickle`, which step 2
reads and resolves. Nothing here reads `exports/`, so this notebook is re-runnable
against the source alone and cannot be invalidated by republishing the dataset.

The historical `prepare_suo.ipynb` that shipped beside the source data is where
every parameter below comes from, including the subset rule.

Run this notebook once. Re-run it only when the source data, the graph parameters,
or the cell metadata change.


## Environment

The interpreter needs `scanpy` and `umap-learn`. Nothing here writes inside the
source data directory.


In [1]:
import gc
import os
import resource
import sys
import time
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

sc.settings.verbosity = 1

# scanpy reaches for tqdm's notebook frontend, which warns when ipywidgets is
# absent. It concerns progress-bar rendering only, and the warning text carries
# the interpreter's own install path into the committed output.
warnings.filterwarnings("ignore", message="IProgress not found")

TIMINGS: dict[str, float] = {}

# getrusage reports the high-water mark in bytes on macOS and in kibibytes
# everywhere else.
RSS_UNIT = 1 if sys.platform == "darwin" else 1024


def peak_rss_gb() -> float:
    """Peak resident set size of this kernel so far, in GB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * RSS_UNIT / 1e9


@contextmanager
def stage(name: str):
    """Record the wall-clock cost of one preparation stage."""
    start = time.perf_counter()
    yield
    TIMINGS[name] = time.perf_counter() - start
    print(f"[{name}] {TIMINGS[name]:.1f} s  peak RSS {peak_rss_gb():.2f} GB")


ANNDATA_VERSION = version("anndata")
SCANPY_VERSION = version("scanpy")
NUMPY_VERSION = version("numpy")

print("anndata", ANNDATA_VERSION)
print("scanpy ", SCANPY_VERSION)
print("numpy  ", NUMPY_VERSION)


anndata 0.12.19
scanpy  1.11.5
numpy   1.26.4


## Paths

Both notebooks locate the repository by walking up for `exports/datasets.json`,
so they run from any working directory.

- **Source** - `.anndatas/suo.h5ad` beside the repository, read-only.
  Override with `CELLUCID_ANNDATA_DIR`.
- **Intermediate** - written *outside every repository*. It carries the full
  count matrix and is far too large to track in git. Override with
  `CELLUCID_INTERMEDIATE_DIR`.


In [2]:
def find_repository_root(start: Path) -> Path:
    """Return the cellucid-datasets checkout that contains `start`."""
    for candidate in [start, *start.parents]:
        if (candidate / "exports" / "datasets.json").is_file():
            return candidate
    raise RuntimeError(f"No cellucid-datasets checkout above {start}")


DATASET_ID = "suo"

REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
WORKSPACE_ROOT = REPOSITORY_ROOT.parent

SOURCE_DIR = Path(os.environ.get("CELLUCID_ANNDATA_DIR", WORKSPACE_ROOT / ".anndatas"))
INTERMEDIATE_DIR = Path(
    os.environ.get("CELLUCID_INTERMEDIATE_DIR", WORKSPACE_ROOT / ".anndatas-intermediate")
)

SOURCE_FILE = SOURCE_DIR / f"{DATASET_ID}.h5ad"
INTERMEDIATE_FILE = INTERMEDIATE_DIR / f"{DATASET_ID}_prepared.h5ad"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)


def show(path: Path) -> str:
    """Render a path relative to the workspace so committed output is portable."""
    try:
        return str(path.relative_to(WORKSPACE_ROOT))
    except ValueError:
        return str(path)


print("repository  :", show(REPOSITORY_ROOT))
print("source      :", show(SOURCE_FILE), f"({SOURCE_FILE.stat().st_size / 1e6:.1f} MB)")
print("intermediate:", show(INTERMEDIATE_FILE))


repository  : cellucid-datasets
source      : .anndatas/suo.h5ad (2356.9 MB)
intermediate: .anndatas-intermediate/suo_prepared.h5ad


## Graph and embedding parameters

One seed and one neighbour count govern the graph and all three UMAPs. These are
not free choices: they are the values the original build used, read out of the
historical `prepare_suo.ipynb` that shipped beside the source data.
Changing any of them changes the published layout.

`SUBSET_FIELD`/`SUBSET_VALUE` are the one line that differs from the `he`,
`garcia` and `kanemaru` builds: this dataset publishes one lineage rather than the
whole source file.


In [3]:
LATENT_KEY = "scanvi"      # scANVI latent representation shipped with the source
N_NEIGHBORS = 15           # neighbours in the one shared kNN graph
MIN_DIST = 0.5             # UMAP min_dist, identical for 1D/2D/3D
RANDOM_SEED = 0            # governs the graph and every embedding

SUBSET_FIELD = "LVL0"                     # the published dataset is one lineage
SUBSET_VALUE = "Haematopoeitic_lineage"   # the source's own spelling, kept verbatim
EXPECTED_N_CELLS = 561_947                # cells the rule must select

UMAP_DIMENSION_KEYS = {1: "X_umap_1d", 2: "X_umap_2d", 3: "X_umap_3d"}


## Load the source

`suo.h5ad` is the `suo_developmental_complete.h5ad` produced by the scVI/scANVI
integration pipeline; it is a derivative, not an archive download. Its public
upstream is ArrayExpress study
[E-MTAB-11341](https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-11341),
which is also the accession `suo_2_export_cellucid.ipynb` publishes as this
dataset's source. The integrated file itself is not distributed, and the copy
that places it under `.anndatas/` is an internal step. To reproduce this
notebook from public data, start at the accession and run your own integration;
to use the result without re-running anything, read `exports/suo/`, which is
tracked in this repository.

The historical notebook wrote its subset and embeddings into a second file,
`suo_developmental_complete_with_3d_umap.h5ad`, which was never copied here;
this notebook rebuilds both from the complete file instead, which is why nothing
downstream depends on the missing one.

The cell also records the per-column bounds of the **whole file**, before any
subsetting. They cost one pass over the matrix and they are what turns the
subset section below into evidence rather than an assertion about itself.


In [4]:
def column_bounds(matrix: sp.csr_matrix) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per-column (minimum, maximum, detected-in) exactly as a quantizing writer sees them.

    A column's minimum is 0 unless the feature is detected in every cell, because
    the implicit zeros of a sparse matrix are real measurements. This is the same
    rule `prepare` applies, computed straight off the CSR arrays so that no
    column-major copy of a 98-million-nonzero matrix is ever materialized.
    """
    n_rows, n_columns = matrix.shape
    detected = np.bincount(matrix.indices, minlength=n_columns)
    high = np.zeros(n_columns, dtype=np.float64)
    np.maximum.at(high, matrix.indices, matrix.data)
    low = np.full(n_columns, np.inf, dtype=np.float64)
    np.minimum.at(low, matrix.indices, matrix.data)
    return np.where(detected == n_rows, low, 0.0), high, detected


with stage("load_source"):
    source = ad.read_h5ad(SOURCE_FILE)

print(source)
print()
print("cells:", f"{source.n_obs:,}", " features:", f"{source.n_vars:,}")

counts = source.X
assert sp.issparse(counts), "expected a sparse count matrix"
assert np.all(counts.data == np.round(counts.data)), "X is not integer-valued"
print(
    "X: sparse counts,",
    f"nnz={counts.nnz:,}",
    f"({counts.nnz / (source.n_obs * source.n_vars):.2%} dense),",
    f"max={counts.data.max():.0f}",
)

with stage("whole_file_column_bounds"):
    whole_file_low, whole_file_high, _ = column_bounds(counts)

print()
print("obsm:", {key: value.shape[1] for key, value in source.obsm.items()})
print("obsp:", list(source.obsp.keys()) or "(empty - this file ships no neighbour graph)")
assert len(source.obsp) == 0, (
    "suo is expected to ship no neighbour graph; if one appears here the "
    "recompute below has something to be compared against"
)


[load_source] 2.1 s  peak RSS 1.70 GB
AnnData object with n_obs × n_vars = 841922 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', '_scvi_batch', '_scvi_labels', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'rank_genes_groups'
    obsm: 'Unintegrated', 'X_pca', 'harmony', 'invae', 'scanvi', 'scvi', 'tardis_1', 'tardis_2'

cells: 841,922  features: 8,192


X: sparse counts, nnz=98,285,588 (1.43% dense), max=9387


[whole_file_column_bounds] 0.6 s  peak RSS 2.38 GB

obsm: {'Unintegrated': 50, 'X_pca': 50, 'harmony': 50, 'invae': 20, 'scanvi': 24, 'scvi': 24, 'tardis_1': 48, 'tardis_2': 40}
obsp: (empty - this file ships no neighbour graph)


## Expression values: raw counts, deliberately

`prepare` receives the untransformed integer count matrix. The historical notebook
calls `sc.pp.normalize_total` and `sc.pp.log1p`, but it applies them to a *second*
AnnData object that it then never uses, and the export it produced carries integer
count maxima throughout.

The rebuild keeps the raw integer counts because that is what every other
generation in this catalog contains. Switching the catalog to depth-normalized
expression is a defensible change, but it is a decision about all five datasets at
once.


## The published subset: one lineage, applied first

`.anndatas/suo.h5ad` holds 841,922 cells and the published dataset holds
561,947. The difference is not a quality filter: it is the single `LVL0`
category `Haematopoeitic_lineage`, which contains exactly that many cells. The
other four lineages - Stromal, Epithelial, Endothelial, Neuronal_and_Glial - are
not in the published dataset.

The historical `prepare_suo.ipynb` applies it as
`adata[adata.obs["LVL0"] == "Haematopoeitic_lineage"].copy()` **before**
`sc.pp.neighbors`, so the graph and all three layouts are built on the subset
alone. Doing it in that order is not an optimization, it is the recipe: a graph
over all 841,922 cells would place the haematopoietic cells differently.

This notebook is the only surviving record of the rule. The export format
publishes no cell barcodes and no source row indices, so `exports/suo/` cannot
say which cells it contains; losing this cell means losing the ability to
rebuild the dataset at all.


In [5]:
lineage = source.obs[SUBSET_FIELD]
for label, count in lineage.value_counts().items():
    marker = "   <- published" if label == SUBSET_VALUE else ""
    print(f"  {label:<22} {count:>9,}{marker}")

selected = (lineage == SUBSET_VALUE).to_numpy()
print()
print(f"{SUBSET_FIELD} == {SUBSET_VALUE!r} selects "
      f"{int(selected.sum()):,} of {source.n_obs:,} cells")
assert int(selected.sum()) == EXPECTED_N_CELLS, (
    f"the subset rule no longer selects the recorded cell count: "
    f"{int(selected.sum()):,} != {EXPECTED_N_CELLS:,}"
)

with stage("subset"):
    subset_counts = counts[selected]
    subset_latent = np.asarray(source.obsm[LATENT_KEY][selected], dtype=np.float32)
    obs = source.obs[selected].copy()

print()
print(f"counts : {counts.shape} -> {subset_counts.shape}, "
      f"nnz {counts.nnz:,} -> {subset_counts.nnz:,}")
print(f"latent : {LATENT_KEY} {source.obsm[LATENT_KEY].shape} -> {subset_latent.shape}")

# The subset is not a relabelling of the same measurements: it changes what the
# gene axis contains. Measuring both bound sets here is what turns the rule above
# into evidence: features whose bounds move, and features the subset empties
# outright, exist only because the subset was applied.
with stage("subset_column_bounds"):
    subset_low, subset_high, subset_detected = column_bounds(subset_counts)

differing = int((~((whole_file_low == subset_low) & (whole_file_high == subset_high))).sum())
print(f"\n{differing:,} of {source.n_vars:,} features have different bounds inside the "
      f"subset than over the whole file")
print(f"features detected in no cell of the subset: "
      f"{int((subset_detected == 0).sum())}")


  Haematopoeitic_lineage   561,947   <- published
  Stromal                  209,775
  Epithelial                35,148
  Endothelial               33,042
  Neuronal_and_Glial         2,010

LVL0 == 'Haematopoeitic_lineage' selects 561,947 of 841,922 cells
[subset] 0.1 s  peak RSS 2.72 GB

counts : (841922, 8192) -> (561947, 8192), nnz 98,285,588 -> 51,729,110
latent : scanvi (841922, 24) -> (561947, 24)


[subset_column_bounds] 0.3 s  peak RSS 3.14 GB

3,735 of 8,192 features have different bounds inside the subset than over the whole file
features detected in no cell of the subset: 4


## Cell metadata

The edits the exported dataset depends on, and nothing else:

- `_scvi_batch` and `_scvi_labels` are scVI bookkeeping. They are integer codes
  with no meaning to a reader and are dropped.
- Subsetting to one lineage empties whole categories, and an unused category
  would be published as a legend entry no cell can ever have. `AnnData` prunes
  them for you; this build slices the frames directly, so the same tidy-up is
  done explicitly and its effect printed.
- `age` is stored as a *categorical* of numeric strings. Post-conception weeks are
  an ordered quantity, so it is converted to a float and exported as a continuous
  field; leaving it categorical would forbid ordering, ranges and a continuous
  colour map, and would sort week 10 before week 6.
- A category label that is padded with whitespace has no glyph the legend can
  draw, and `cellucid.prepare` rejects one rather than repairing it: trimming can
  merge two categories and move cells between them without saying so. The decision
  therefore belongs to the dataset build. The scan below runs over every
  categorical field, proves that trimming merges nothing before applying it, and
  renames the *categories* rather than rewriting the values, so every cell keeps
  the code it already had.

Everything else is passed through unchanged.


In [6]:
dropped = [column for column in ("_scvi_batch", "_scvi_labels") if column in obs]
obs = obs.drop(columns=dropped)
print("dropped:", dropped)

print("\ncategories emptied by the subset:")
for column in [c for c in obs.columns if isinstance(obs[c].dtype, pd.CategoricalDtype)]:
    before = obs[column].cat.categories.size
    obs[column] = obs[column].cat.remove_unused_categories()
    after = obs[column].cat.categories.size
    if before != after:
        print(f"  {column:<36} {before:>3} -> {after:>3}")

age_labels = obs["age"].cat.categories.tolist()
obs["age"] = obs["age"].astype(str).astype(np.float64)
assert obs["age"].notna().all(), "age must convert cleanly to a number"
print(f"\nage: categorical {age_labels} -> continuous "
      f"[{obs['age'].min()}, {obs['age'].max()}]")

categorical = [c for c in obs.columns if isinstance(obs[c].dtype, pd.CategoricalDtype)]
padded = {
    column: [label for label in obs[column].cat.categories
             if isinstance(label, str) and label != label.strip()]
    for column in categorical
}
padded = {column: labels for column, labels in padded.items() if labels}
print(f"\ncategorical fields carrying padded labels: "
      f"{ {column: len(labels) for column, labels in padded.items()} or 'none'}")

for column in padded:
    source_labels = obs[column].cat.categories.tolist()
    trimmed = [label.strip() for label in source_labels]
    # Trimming is only honest if it merges nothing. Two labels collapsing to one
    # would move cells between categories, which is a change to the annotation
    # rather than to its presentation, and this build must not make it silently.
    assert len(set(trimmed)) == len(set(source_labels)), (
        f"stripping {column!r} merges categories and would move cells between them: "
        f"{sorted(set(source_labels))}"
    )
    codes_before = obs[column].cat.codes.to_numpy().copy()
    obs[column] = obs[column].cat.rename_categories(trimmed)
    assert np.array_equal(obs[column].cat.codes.to_numpy(), codes_before), (
        f"trimming {column!r} moved cells between categories"
    )
    print(f"  {column}: {len(padded[column])} of {len(source_labels)} labels trimmed, "
          f"every cell keeps its code")
    print(f"    before {source_labels}")
    print(f"    after  {trimmed}")

LABEL_NORMALIZATION = ", ".join(
    f"{column}: trailing whitespace stripped from {len(labels)} of "
    f"{obs[column].cat.categories.size} labels"
    for column, labels in padded.items()
) or "none"

continuous = [c for c in obs.columns if not isinstance(obs[c].dtype, pd.CategoricalDtype)]
categorical = [c for c in obs.columns if isinstance(obs[c].dtype, pd.CategoricalDtype)]
print(f"\n{len(obs.columns)} obs fields: "
      f"{len(continuous)} continuous {continuous}, {len(categorical)} categorical")
print("largest category count:",
      max(obs[c].cat.categories.size for c in categorical))


dropped: ['_scvi_batch', '_scvi_labels']

categories emptied by the subset:
  cell_type                             91 ->  69
  LVL3                                  91 ->  69
  LVL2                                  34 ->  16
  LVL1                                  19 ->   4
  LVL0                                   5 ->   1

age: categorical ['10.0', '11.0', '12.0', '13.0', '14.0', '15.0', '16.0', '17.0', '4.0', '7.0', '8.0', '9.0'] -> continuous [4.0, 17.0]

categorical fields carrying padded labels: {'organ': 8}
  organ: 8 of 9 labels trimmed, every cell keeps its code
    before ['Bone_marrow ', 'Gut ', 'Kidney ', 'Liver ', 'Mesenteric_lymph_node', 'Skin ', 'Spleen ', 'Thymus ', 'Yolk_sac ']
    after  ['Bone_marrow', 'Gut', 'Kidney', 'Liver', 'Mesenteric_lymph_node', 'Skin', 'Spleen', 'Thymus', 'Yolk_sac']

16 obs fields: 2 continuous ['age', 'n_genes'], 14 categorical
largest category count: 69


## Feature identity: positional here, named in step 2

**`.anndatas/suo.h5ad` carries no gene names.** Its `var` index is the
positional matrix column number, `'0'` ... `'8191'`, and `var` has no other column.
The integration pipeline held the identifiers in a separate `var` frame, which is
`.anndatas/varnames/dataset_complete_Suo_varnames.pickle`.

That frame is read in **step 2**, together with the one permitted symbol mapping,
because naming a gene is a decision about what the viewer prints and not about how
the cells are laid out. Keeping it there means a change to the naming costs one
cheap re-run of step 2 rather than a re-embedding, and it keeps this notebook
dependent on the source file alone.

What this notebook does record is the positional index itself, in
`var['source_var_index']`, so that step 2's alignment of the identifier frame to
the matrix columns is checkable rather than assumed.


In [7]:
var = source.var.copy()
assert list(var.columns) == [], f"source var unexpectedly carries columns: {list(var.columns)}"
assert var.index.tolist() == [str(i) for i in range(source.n_vars)], (
    "source var index is not the positional matrix column number - step 2 aligns "
    "the identifier frame to it positionally and can no longer do so"
)

var["source_var_index"] = np.arange(source.n_vars, dtype=np.int64)
var.index = var.index.astype(str)
var.index.name = "source_var_position"

FEATURE_IDENTITY = "positional_source_index; gene names resolved in step 2"
print("feature identity:", FEATURE_IDENTITY)
print("features:", f"{var.shape[0]:,}", " first:", var.index[:3].tolist(),
      " last:", var.index[-3:].tolist())


feature identity: positional_source_index; gene names resolved in step 2
features: 8,192  first: ['0', '1', '2']  last: ['8189', '8190', '8191']


## One shared kNN graph

Unlike its siblings there is nothing here to reuse and nothing to compare
against: `.anndatas/suo.h5ad` has an empty `obsp` and no `uns['neighbors']`.
`he`, `garcia` and `kanemaru` each ship a RAPIDS graph that this build does not
use; `suo` ships none at all.

So the parameters come entirely from the historical `prepare_suo.ipynb` -
`sc.pp.neighbors(n_neighbors=15, random_state=0, use_rep='scanvi')`, called on
the subset, before any UMAP.

Above 4,096 cells scanpy's neighbour search is approximate, so this graph is one
deterministic draw from an approximate search rather than an exact kNN graph.


In [8]:
adata = ad.AnnData(
    X=subset_counts,
    obs=obs,
    var=var,
    obsm={LATENT_KEY: subset_latent},
)
del source, counts, subset_counts, subset_latent, lineage
gc.collect()
print(f"source released; peak RSS so far {peak_rss_gb():.2f} GB")

with stage("neighbors"):
    sc.pp.neighbors(
        adata,
        n_neighbors=N_NEIGHBORS,
        use_rep=LATENT_KEY,
        random_state=RANDOM_SEED,
    )

connectivities = adata.obsp["connectivities"].tocsr()
connectivities.eliminate_zeros()
assert (abs(connectivities - connectivities.T) > 0).nnz == 0, "graph must be symmetric"
assert connectivities.diagonal().sum() == 0, "graph must have a zero diagonal"
assert connectivities.data.min() > 0, "graph must not store zero weights"
adata.obsp["connectivities"] = connectivities

print("recomputed graph:", dict(adata.uns["neighbors"]["params"]))
print(f"edges (stored, both directions): {connectivities.nnz:,}"
      f"  undirected: {connectivities.nnz // 2:,}")


source released; peak RSS so far 3.29 GB


[neighbors] 52.9 s  peak RSS 3.31 GB
recomputed graph: {'n_neighbors': 15, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean', 'use_rep': 'scanvi'}
edges (stored, both directions): 12,558,296  undirected: 6,279,148


## 1D, 2D and 3D UMAP from that one graph

Each embedding is laid out on a throwaway `AnnData` that carries the same
`obsp`/`uns['neighbors']` pair, so `sc.tl.umap` never rebuilds the graph and the
three layouts stay mutually consistent. Only the number of components differs.


In [9]:
def umap_from_shared_graph(host: ad.AnnData, n_components: int) -> np.ndarray:
    """Lay out the already-computed neighbour graph in `n_components` dimensions."""
    scratch = ad.AnnData(
        obs=host.obs[[]],
        obsp={
            "connectivities": host.obsp["connectivities"],
            "distances": host.obsp["distances"],
        },
        obsm={LATENT_KEY: host.obsm[LATENT_KEY]},
        uns={"neighbors": dict(host.uns["neighbors"])},
    )
    sc.tl.umap(
        scratch,
        n_components=n_components,
        min_dist=MIN_DIST,
        random_state=RANDOM_SEED,
    )
    embedding = np.asarray(scratch.obsm["X_umap"], dtype=np.float32)
    del scratch
    gc.collect()
    return embedding


for n_components, key in UMAP_DIMENSION_KEYS.items():
    with stage(f"umap_{n_components}d"):
        adata.obsm[key] = umap_from_shared_graph(adata, n_components)

for n_components, key in UMAP_DIMENSION_KEYS.items():
    coords = adata.obsm[key]
    assert coords.shape == (adata.n_obs, n_components)
    assert np.isfinite(coords).all()
    spans = coords.max(axis=0) - coords.min(axis=0)
    assert spans.min() > 0, f"{key} collapsed to a point"
    print(f"{key}: shape {coords.shape}, span {np.round(spans, 2).tolist()}")


[umap_1d] 494.1 s  peak RSS 4.63 GB


[umap_2d] 425.0 s  peak RSS 5.52 GB


[umap_3d] 430.2 s  peak RSS 5.84 GB
X_umap_1d: shape (561947, 1), span [66.4800033569336]
X_umap_2d: shape (561947, 2), span [27.34000015258789, 28.729999542236328]
X_umap_3d: shape (561947, 3), span [19.5, 20.979999542236328, 21.139999389648438]


## Write the intermediate

The intermediate holds exactly what step 2 consumes and nothing else: counts,
curated `obs`, the positional feature index, the latent representation, the three
embeddings, the shared graph, and a provenance record.

`created_at` is fixed here and reused by step 2, so re-running the export after an
API change reproduces the same `dataset_identity.json` timestamp instead of
inventing a new one.


In [10]:
CREATED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

adata.uns = {
    "cellucid_build": {
        "dataset_id": DATASET_ID,
        "created_at": CREATED_AT,
        "source_file": SOURCE_FILE.name,
        "source_bytes": int(SOURCE_FILE.stat().st_size),
        "subset_rule": f"obs[{SUBSET_FIELD!r}] == {SUBSET_VALUE!r}",
        "subset_n_cells": int(adata.n_obs),
        "latent_key": LATENT_KEY,
        "n_neighbors": int(N_NEIGHBORS),
        "min_dist": float(MIN_DIST),
        "random_state": int(RANDOM_SEED),
        "feature_identity": FEATURE_IDENTITY,
        "label_normalization": LABEL_NORMALIZATION,
        "expression": "raw integer counts, untransformed",
        "anndata_version": ANNDATA_VERSION,
        "scanpy_version": SCANPY_VERSION,
        "numpy_version": NUMPY_VERSION,
    }
}

with stage("write_intermediate"):
    adata.write_h5ad(INTERMEDIATE_FILE, compression="gzip")

print(adata)
print()
print("wrote", show(INTERMEDIATE_FILE), f"({INTERMEDIATE_FILE.stat().st_size / 1e6:.1f} MB)")
print("created_at", CREATED_AT)


[write_intermediate] 12.2 s  peak RSS 5.84 GB
AnnData object with n_obs × n_vars = 561947 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    var: 'source_var_index'
    uns: 'cellucid_build'
    obsm: 'scanvi', 'X_umap_1d', 'X_umap_2d', 'X_umap_3d'
    obsp: 'distances', 'connectivities'

wrote .anndatas-intermediate/suo_prepared.h5ad (336.7 MB)
created_at 2026-07-31T11:34:27Z


## Measured cost

Wall-clock per stage and the peak resident set size on the machine that produced
the committed outputs.


In [11]:
total = sum(TIMINGS.values())
for name, seconds in TIMINGS.items():
    print(f"{name:<26} {seconds:8.1f} s")
print(f"{'TOTAL':<26} {total:8.1f} s")
print(f"{'peak RSS':<26} {peak_rss_gb():8.2f} GB")


load_source                     2.1 s
whole_file_column_bounds        0.6 s
subset                          0.1 s
subset_column_bounds            0.3 s
neighbors                      52.9 s
umap_1d                       494.1 s
umap_2d                       425.0 s
umap_3d                       430.2 s
write_intermediate             12.2 s
TOTAL                        1417.5 s
peak RSS                       5.84 GB


Next: run `suo_2_export_cellucid.ipynb`. It reads only
`<intermediate>/suo_prepared.h5ad`, resolves the gene names, and never
recomputes an embedding.
